# 원본 Gemma 4 LiveCodeBench 175문항

`putare08-source-*.zip`과 `putare08-data-*.zip`을 `MyDrive/Putare`에 올리세요. 설정 플래그를 필요한 단계만 켜고 위에서 아래로 실행합니다. 기본 Run all은 설치·다운로드·추론·채점을 실행하지 않습니다.

모델은 Hugging Face에서 런타임 로컬 디스크로 다운로드하며 128 experts 원본과 해시를 확인합니다. Google Drive에 각 완료 문제와 전체 라우팅 기록을 필수 백업합니다. 백업이 실패하면 중단합니다. 새 런타임에서 복원하려면 `RESTORE_CHECKPOINT=True`를 켜고 빈 로컬 실행 경로에 복원한 후 생성 셀을 다시 실행하세요.

재개에는 모델·소스·설정·엔진 파일의 해시와 경로가 기존 실행과 같아야 합니다. 새 런타임에서 엔진을 다시 빌드한 결과가 다르면 재개가 거부되며, 기존 백업을 보존한 채 별도 실행 경로와 백업 경로를 사용해야 합니다.

모든 토큰의 30개 층에서 선택된 Expert ID와 gate 가중치를 저장하므로 디스크와 Drive 사용량이 매우 커질 수 있습니다. 사용량 경고는 강제 메모리 제한이 아닙니다. 실제 RTX PRO 6000 CUDA 동작은 아직 검증되지 않았습니다. 공식 채점은 별도 프로세스에서 실행되지만 보안 격리가 없으므로 생성 코드에 현재 런타임 사용자 권한이 있습니다.


In [1]:
SETUP_COLAB = True
DOWNLOAD_MODEL = True
RESTORE_CHECKPOINT = False
RUN_GENERATION = True
RUN_GRADING = True
SOURCE_ZIP = '/content/drive/MyDrive/Putare/putare08-source-d698b3315ac2.zip'
DATA_ZIP = '/content/drive/MyDrive/Putare/putare08-data-675fe519d982.zip'
MODEL_PATH = '/content/models/gemma-4-26B-A4B-it-UD-IQ4_NL.gguf'
CHECKPOINT_DIR = '/content/drive/MyDrive/Putare/runs/gemma4-base-lcb175-001'


In [2]:
import hashlib, json, os, shutil, sys, zipfile
from pathlib import Path, PurePosixPath

def sha256_file(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def restore_archives(root, archives):
    """모든 ZIP과 기존 경로를 먼저 검증하고 누락된 파일만 복원한다."""
    # 2026-11-09: 불완전한 전송과 경로 탈출·기존 데이터 덮어쓰기를 함께 방지한다.
    root = Path(root)
    if root.is_symlink():
        raise ValueError('복원 위치가 심볼릭 링크입니다')
    validated = []
    used = set()
    try:
        for archive_path, expected_hash in archives:
            archive_path = Path(archive_path)
            if not archive_path.is_file():
                raise FileNotFoundError(f'Drive에 ZIP을 업로드하세요: {archive_path}')
            if sha256_file(archive_path) != expected_hash:
                raise ValueError(f'ZIP SHA256 불일치: {archive_path}')
            archive = zipfile.ZipFile(archive_path)
            validated.append((archive, {}))
            names = archive.namelist()
            if len(names) != len(set(names)) or '_putare_inventory.json' not in names:
                raise ValueError('ZIP 파일 목록에 중복 또는 인벤토리 누락')
            inventory = json.loads(archive.read('_putare_inventory.json'))
            if not isinstance(inventory, dict) or set(names) != set(inventory) | {'_putare_inventory.json'}:
                raise ValueError('ZIP 인벤토리 불일치')
            for name, digest in inventory.items():
                path = PurePosixPath(name)
                if (not isinstance(name, str) or path.is_absolute() or not path.parts
                        or any(p in ('', '.', '..') for p in name.split('/')) or '\\' in name or ':' in name):
                    raise ValueError('안전하지 않은 ZIP 경로')
                info = archive.getinfo(name)
                if info.is_dir() or (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError('ZIP에 링크 또는 디렉터리가 있습니다')
                if name in used:
                    raise ValueError('소스와 데이터 ZIP에 중복 경로가 있습니다: ' + name)
                used.add(name)
                with archive.open(info) as stream:
                    if hashlib.file_digest(stream, 'sha256').hexdigest() != digest:
                        raise ValueError('ZIP 파일 SHA256 불일치: ' + name)
                target = root.joinpath(*path.parts)
                for part in [root, *[root.joinpath(*path.parts[:i]) for i in range(1, len(path.parts) + 1)]]:
                    if part.is_symlink():
                        raise ValueError('복원 경로에 심볼릭 링크가 있습니다: ' + str(part))
                    if part != target and part.exists() and not part.is_dir():
                        raise FileExistsError('복원 폴더 위치가 파일입니다: ' + str(part))
                if target.exists() and (not target.is_file() or sha256_file(target) != digest):
                    raise FileExistsError('기존 파일이 달라 덮어쓰지 않습니다: ' + str(target))
                validated[-1][1][name] = target
        for archive, paths in validated:
            for name, target in paths.items():
                if not target.exists():
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(name) as stream, target.open('xb') as output:
                        shutil.copyfileobj(stream, output)
    except (zipfile.BadZipFile, KeyError, UnicodeError) as error:
        raise ValueError('ZIP이 손상되었습니다') from error
    finally:
        for archive, _ in validated:
            archive.close()
    return len(used)


if SETUP_COLAB:
    _PUTARE_SETUP_READY = False
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.ismount('/content/drive'):
        raise RuntimeError('Google Drive가 마운트되지 않았습니다')
    ROOT = Path('/content/Putare') / ('d698b3315ac220e400fe0e08204429b03d7da1c3eda4a383bf80656b9137f1e8'[:12] + '-' + '675fe519d982ad016106b6babf0ad1dc6b523251c3197afcbe8998feadb2f22e'[:12])
    for _name, _module in list(sys.modules.items()):
        if _name not in ('tools', 'moe_prune') and not _name.startswith(('tools.', 'moe_prune.')):
            continue
        _file = getattr(_module, '__file__', None)
        _paths = getattr(_module, '__path__', ()) if not _file else ()
        if (_file and not Path(_file).resolve().is_relative_to(ROOT.resolve())) or any(not Path(p).resolve().is_relative_to(ROOT.resolve()) for p in _paths):
            raise RuntimeError('다른 Putare 버전이 로드되었습니다. 커널을 다시 시작하세요: ' + _name)
    restore_archives(ROOT, [(SOURCE_ZIP, 'd698b3315ac220e400fe0e08204429b03d7da1c3eda4a383bf80656b9137f1e8'), (DATA_ZIP, '675fe519d982ad016106b6babf0ad1dc6b523251c3197afcbe8998feadb2f22e')])
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
if SETUP_COLAB:
    _PUTARE_SETUP_READY = False
    from tools.colab_setup import prepare_environment
    from tools import benchmark175
    _PUTARE_SETUP_STATE = prepare_environment(ROOT, globals().get('_PUTARE_SETUP_STATE'))
    def _validate_backup():
        if not os.path.ismount('/content/drive'):
            raise RuntimeError('Google Drive 연결을 확인하세요')
        if not Path(CHECKPOINT_DIR).resolve().is_relative_to(Path('/content/drive/MyDrive').resolve()):
            raise ValueError('백업 경로는 마운트된 MyDrive 아래여야 합니다')
        benchmark175.preflight_checkpoint(CHECKPOINT_DIR)
    _validate_backup()
    ENGINE = ROOT / '.cache/llama-build/bin/llama-expert-research'
    RUN_DIR = ROOT / 'runs/gemma4-base-lcb175-001'
    MODEL = Path(MODEL_PATH)
    _PUTARE_SETUP_READY = True
    print('준비 완료:', RUN_DIR)


GPU: NVIDIA A100-SXM4-80GB, Compute Capability: 80
공식 LCB commit 28fef95ea8c9f7a547c8329f2cd3d32b92c1fa24, venv /content/Putare/d698b3315ac2-675fe519d982/.cache/putare-lcb/venv, source /content/Putare/d698b3315ac2-675fe519d982/.cache/putare-lcb/livecodebench/28fef95ea8c9f7a547c8329f2cd3d32b92c1fa24
준비 완료: /content/Putare/d698b3315ac2-675fe519d982/runs/gemma4-base-lcb175-001


In [4]:
if DOWNLOAD_MODEL:
    if not globals().get('_PUTARE_SETUP_READY', False):
        raise RuntimeError('먼저 환경 준비를 완료하세요')
    print('모델 준비:', benchmark175.download_base_model(Path(MODEL_PATH)))


gemma-4-26B-A4B-it-UD-IQ4_NL.gguf: reconstructing file:   0%|          |  0.00B / 13.6GB            

gemma-4-26B-A4B-it-UD-IQ4_NL.gguf: downloading bytes:           |  0.00B            

모델 준비: /content/models/gemma-4-26B-A4B-it-UD-IQ4_NL.gguf


In [5]:
if RESTORE_CHECKPOINT:
    if not globals().get('_PUTARE_SETUP_READY', False):
        raise RuntimeError('먼저 환경 준비를 완료하세요')
    _validate_backup()
    print('백업 복원:', benchmark175.restore_checkpoint(CHECKPOINT_DIR, RUN_DIR))


In [ ]:
if RUN_GENERATION:
    if not globals().get('_PUTARE_SETUP_READY', False):
        raise RuntimeError('먼저 환경 준비를 완료하세요')
    _validate_backup()
    print(benchmark175.generate(RUN_DIR, model=Path(MODEL_PATH), engine=ENGINE, checkpoint_dir=CHECKPOINT_DIR, model_profile='base'))


1/175 livecodebench:3674
2/175 livecodebench:3677
3/175 livecodebench:3687
4/175 livecodebench:3692
5/175 livecodebench:3696
6/175 livecodebench:3697
7/175 livecodebench:3701
8/175 livecodebench:3704
9/175 livecodebench:3705
10/175 livecodebench:3708
11/175 livecodebench:3709
12/175 livecodebench:3714
13/175 livecodebench:3717
14/175 livecodebench:3720
15/175 livecodebench:3721
16/175 livecodebench:3722
17/175 livecodebench:3723
18/175 livecodebench:3725
19/175 livecodebench:3731
20/175 livecodebench:3733
21/175 livecodebench:3736
22/175 livecodebench:3737
23/175 livecodebench:3739
24/175 livecodebench:3743
25/175 livecodebench:3744
26/175 livecodebench:3747
27/175 livecodebench:3748
28/175 livecodebench:3750
29/175 livecodebench:3751
30/175 livecodebench:3753
31/175 livecodebench:3754
32/175 livecodebench:3759
33/175 livecodebench:3760
34/175 livecodebench:3762
35/175 livecodebench:3763
36/175 livecodebench:3764
37/175 livecodebench:3765
38/175 livecodebench:3768
39/175 livecodebench:

In [ ]:
if RUN_GRADING:
    if not globals().get('_PUTARE_SETUP_READY', False):
        raise RuntimeError('먼저 환경 준비를 완료하세요')
    _validate_backup()
    summary = benchmark175.grade(RUN_DIR, model=Path(MODEL_PATH), engine=ENGINE, checkpoint_dir=CHECKPOINT_DIR, model_profile='base')
    print(f"점수: {summary['passed']}/175" if summary.get('complete') else '채점 미완료: 인프라 오류 또는 시간 초과를 확인하세요')
